## Project: Interview Question Creator — RAG-Based Question & Answer Generation

### Phase 1 — Experimentation (Jupyter Notebook)

### 1. Objective
- The objective of Phase 1 is to experimentally develop and validate the core components of the **Interview Question Creator** before moving to the production GenAI pipeline.
In this phase, the complete RAG-based workflow is implemented and tested inside a Jupyter Notebook. Each individual component is evaluated independently to ensure that document loading, text splitting, embedding generation, vector storage, retrieval, prompt engineering, question generation, answer generation, and structured output work correctly.

In [1]:
print("Okay")

Okay


### 2. Load the Groq API KEY From .env file

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
# save Groq Api Key into our environment 
os.environ["GROQ_API_KEY"] = GROQ_API_KEY


### 3. Load PDF Document

In [4]:
%pwd

'e:\\Data Science\\ALL Projects\\GenAI Projects\\Interview-Question-Creator\\Notebook'

In [6]:
# To get back one step or go to the Root dir (Interview-Question-Creator)
%cd ..

e:\Data Science\ALL Projects\GenAI Projects\Interview-Question-Creator


In [9]:
from langchain_community.document_loaders import PyPDFLoader


# load the data file from data dir
file_path = "data/SDG.pdf"

loader = PyPDFLoader(file_path)

data = loader.load()

print(f"Length: - {len(data)} Pages")

Length: - 24 Pages


In [11]:
print(f"Data: - \n{data}")

Data: - 
[Document(metadata={'source': 'data/SDG.pdf', 'page': 0}, page_content=''), Document(metadata={'source': 'data/SDG.pdf', 'page': 1}, page_content=''), Document(metadata={'source': 'data/SDG.pdf', 'page': 2}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough  \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew  \nthat earthquakes and floods were inevitable, but that the high death  \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 

In [ ]:
# formatting the data
question_gen = ""

for page in data:
    question_gen += page.page_content

print(f"The Completed Data: \n{question_gen[:200]} ") # First 200 words

The Completed Data: 
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place,  


### 4. Chunk Distribution

#### First Layer of Chunking

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter_ques_gen = RecursiveCharacterTextSplitter(
    chunk_size=10000,
    chunk_overlap=200
)

In [25]:
# chunk of the document(loader)
chunk_ques_gen = splitter_ques_gen.split_text(question_gen)

print(f"Total Chunk: {len(chunk_ques_gen)}")


Total Chunk: 1


In [26]:
chunk_ques_gen

['IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough  \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew  \nthat earthquakes and floods were inevitable, but that the high death  \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15

### Second Layer of Chunking

In [28]:
# Type of First chunking
type(chunk_ques_gen)

list

In [ ]:
# inside the list 
type(chunk_ques_gen[0])

str

- Key Note:
    - We convert string into Document

In [30]:
# Firstly convert first layer of chunking into Document
from langchain_core.documents import Document

document_ques_gen = [Document(page_content = t) for t in chunk_ques_gen]

print(f"Document content: \n {document_ques_gen}")

Document content: 
 [Document(page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough  \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew  \nthat earthquakes and floods were inevitable, but that the high death  \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample ev

In [31]:
# Type of second layer of chunking
type(document_ques_gen)

list

In [32]:
# inside the list 
type(document_ques_gen[0])

langchain_core.documents.base.Document

In [39]:
## Now Apply second layer of Chunking
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter_ans_gen = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

# chunk of the document(loader)
docment_chunk_ans_gen = splitter_ans_gen.split_documents(document_ques_gen)

print(f"Total Chunk: {len(docment_chunk_ans_gen)}")

Total Chunk: 9


In [40]:
print(f"Second Chuk: \n{docment_chunk_ans_gen[1].page_content}")

Second Chuk: 
that’s about the equivalent of the entire population of Europe living in 
extreme poverty. Now it’s time to build on what we learned and end 
poverty altogether. END HUNGER, ACHIEVE FOOD SECURITY 
AND IMPROVED NUTRITION AND PROMOTE 
SUSTAINABLE AGRICUL TURE
In the past 20 years, hunger has dropped by almost half. Many 
countries that used to suffer from famine and hunger can now 
meet the nutritional needs of their most vulnerable people. It’s an 
incredible accomplishment. Now we can go further and end hunger 
and malnutrition once and for all. That means doing things such as 
promoting sustainable agriculture and supporting small farmers. It’s a tall 
order. But for the sake of the nearly 1 out of every 9 people on earth who 
go to bed hungry every night, we’ve got to try. Imagine a world where 
everyone has access to sufficient and nutritious food all year round. 
Together, we can make that a reality by 2030. ENSURE HEAL THY LIVES AND PROMOTE 
WELL-BEING FOR ALL AT ALL

### 5. Initialize Embedding Model and Generate Document Embeddings

In [56]:
## Initialize Embedding Model
# Download from the hugging face platform then initialize the model 
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


c:\Users\kz\anaconda3\envs\llmapp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


### 6. Create a FAISS Vector Database

In [57]:
# Create a FAISS Vector Database
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents = docment_chunk_ans_gen, # this is the entire information that store into FAISS vector Database.
    embedding= embeddings
    
)

### 7. Create an vector store retriever
- retriever is need when we generate answer.

In [ ]:
# Create Retriever
retriever = vectorstore.as_retriever(
    search_type = "similarity", # By Default
    search_kwargs = {
        "k": 1,
    }

)

### 8. Create Question Generation

- 1. Prompt 1: To Create Generic Question
    - chunk_doucument  ---> LLM -----> Generate Generic Questions
    - Generate Generic Question can some mistake.


- 2. Prompt 2: Refine Question
    -  Generate Generic Question  ---> LLM -----> Generate Questions
    - Create Prompt 2 to refine the mistake of Prompt 1.

In [ ]:
# Initialize the LLM Question Generation
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)


#### 1. Prompt 1: To Create Generic Question
- chunk_doucument ---> LLM -----> Generate Generic Questions

- where,
    - Prompt 1: chunk of document.

In [62]:
# Define a Prompt to generate Question
from langchain_core.prompts import PromptTemplate

# Prompt template (Key Note: Specific your tone what are exactly you want to be Create)
prompt_template = """
You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

---------
{text}
---------

Create questions that will prepared the coders or programmers for their test.
Make sure not to lose any important information.

QUESTIONS:

"""

# Now create prompt
PROMPT_QUESTIONS = PromptTemplate(
    template = prompt_template,
    input_variables = ["text"]
)



#### 2. Prompt 2: Refine Question
- Generate Generic Question  ---> LLM -----> Generate Questions

- where,
    - Prompt 2: Generated Generic Question (existing Questions)
    - that also the chunk of documents

In [64]:

# Prompt template 2 (here, inputs: existing question, chunk of documents)

refine_template = ("""
You are an expert at creating practice questions based on coding material and documentation.

Generate interview questions based strictly on the provided context.

Question requirements:
- Questions must be contextual and derived from the provided text.
- Prefer scenario-based, application-oriented, and reasoning-based questions.
- Avoid simple definition-recall questions.
- Avoid questions that can be answered without understanding the provided context.
- Do not invent information outside the provided context.
- Do not include explanations, headings, answers, or refinement notes.

Generate:
- 5 conceptual/contextual questions
- 3 scenario-based questions
- 2 reasoning/application questions

Return ONLY the questions as a numbered list.

Your goal is to help a coder or programmer prepare for a coding test.
We have received some practice questions to a certain extent: {existing_answer}.
We have the option to refine the existing questions or add new ones.
(only if necessary) with some more context below.
------------
{text}
------------

Given the new context on Creative Questions or Contextual Questions, refine the original questions in English.
If the context is not helpful, please provide the original questions.
QUESTIONS:


"""
)

# Now create prompt 2
REFINE_PROMPT_QUESTIONS = PromptTemplate(
    template = refine_template,
    input_variables = ["existing_answer","text"]
)


In [65]:
# we define output parser
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [66]:
from langchain_core.runnables import RunnablePassthrough


# =========================================================
# STEP 1: Generate initial/generic questions
# =========================================================

Generic_Question_chain = (
    PROMPT_QUESTIONS
    | model
    | parser
)


# =========================================================
# STEP 2: Refine the generated questions
#
# The refinement prompt requires:
#   1. existing_answer -> questions generated in Step 1
#   2. text            -> original document text
# =========================================================

Question_Generate_chain = (
    {
        "existing_answer": Generic_Question_chain,
        "text": RunnablePassthrough()
    }
    | REFINE_PROMPT_QUESTIONS
    | model
    | parser
)

In [ ]:
print(f"Question 1: \n{all_questions[0]}")

In [ ]:
""" 
                    ┌── Generic_Question_chain
                    │
document text ──────┤
                    │
                    └── RunnablePassthrough()
                           │
                           ▼
                    ┌─────────────────┐
                    │ existing_answer │
                    │ text            │
                    └────────┬────────┘
                             ▼
                     REFINE_PROMPT
                             ▼
                           LLM
                             ▼
                          Parser
                             ▼
                     Final Questions



"""

In [70]:
# Store the generated questions from all document chunks
all_questions = []


# Process each document chunk separately
for document in document_ques_gen:

    # Extract only the actual text from the LangChain Document
    text = document.page_content

    # Generate questions from this text chunk
    questions = Question_Generate_chain.invoke(text)

    # Store the generated questions
    all_questions.append(questions)

In [71]:
print(f"Question 1: \n{all_questions[0]}")

Question 1: 
1. How does the SDG plan’s emphasis on equitable resource distribution address the issue of food availability mentioned in the passage?  
2. In what ways does the passage link the progress in reducing extreme poverty to the broader goal of ending hunger and malnutrition?  
3. Explain how the SDG targets for clean water and sanitation interact with the projected increase in water scarcity by 2050.  
4. Discuss the role of technological innovation, as described in the passage, in achieving both economic growth and environmental sustainability.  
5. Analyze how the SDG framework seeks to balance the need for industrialization with the protection of terrestrial ecosystems.  

6. Imagine you are given a dataset of countries with their poverty rates and populations. Write a Python function that returns the top five countries that would benefit most from the SDG “End extreme poverty” goal, defined as the highest product of poverty_rate × population.  
7. Suppose you need to extra

### 9. Generate Answers

#### Create Answer Generation Prompt

In [79]:
from langchain_core.prompts import ChatPromptTemplate

# Define the prompt that handles the context documents
system_prompt = (
""" 
You are a technical assistant. Give the shortest correct answer that fully addresses the question. Use bullet points only when listing multiple distinct items; otherwise use plain sentences. No introductory phrases like "Great question!" — start directly with the answer.

Answer the user's question using only the provided context. Respond in 1-3 sentences. If the context doesn't contain the answer, say so in one sentence — do not guess or use outside knowledge. Do not repeat the question back.
Context:\n{context}    

"""
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])


In [94]:
# Create Retriever
retriever = vectorstore.as_retriever(
    search_type = "similarity", # By Default
    search_kwargs = {
        "k": 4,
    }

)

In [95]:
# Helper to format retrieved documents into a single text block
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [96]:
## This is the retrieval response
retrieval_response = retriever.invoke("Explain how the SDG targets for clean water and sanitation interact")

format_retrieval_response = format_docs(retrieval_response)

print(f"Format Retrieval Response: \n{format_retrieval_response}")

Format Retrieval Response: 
number is projected to go even higher as a result of climate change. 
If we continue the path we’re on, by 2050 at least one in four people 
are likely to be affected by recurring water shortages. But we can take 
a new path—more international cooperation, protecting wetlands 
and rivers, sharing water-treatment technologies—that leads to 
accomplishing this Goal. ENSURE AVAILABILITY AND SUSTAINABLE 
MANAGEMENT OF WATER AND SANITATION 
FOR ALL  ENSURE ACCESS TO AFFORDABLE, RELIABLE, 
SUSTAINABLE AND MODERN ENERGY FOR 
ALL 
Between 1990 and 2010, the number of people with access to electricity 
increased by 1.7 billion. That’s progress to be proud of. And yet as the 
world’s population continues to rise, still more people will need cheap 
energy to light their homes and streets, use phones and computers, 
and do their everyday business. How we get that energy is at issue; fossil 
fuels and greenhouse gas emissions are making drastic changes in the 
climate, l

- Note that:
    - Retrieval response are so long . retrieval try to find the similarity response based on the Question.

    - Therefore, we use the LLM to get the refine and organize answer of the provided question.

In [97]:
# Initialize the LLM Answers of the Created Questions
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

In [98]:
# Define a Chain 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# The LCEL pipeline
answer_generation_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)



In [102]:
# # Answer each question and save to a file
# for question in all_questions:
#     print("Question: ", question)
#     answer = answer_generation_chain.invoke(question)
#     print("Answer: ", answer)
#     print("--------------------------------------------------\\n\\n")
    
#     # Save answer to file
#     with open("answers.txt", "a") as f:
#         f.write("Question: " + question + "\\n")
#         f.write("Answer: " + answer + "\\n")
#         f.write("--------------------------------------------------\\n\\n")



### 10. Export Results to file format

In [101]:
for i, question in enumerate(all_questions, start=1):
    print(f"Question {i}: {question}")
    
    answer = answer_generation_chain.invoke(question)
    print(f"Answer {i}: {answer}")
    print("-" * 50, "\n")
    
    # Save in structured, numbered format
    with open("answers.txt", "a", encoding="utf-8") as f:
        f.write(f"Question {i}: {question}\n")
        f.write(f"Answer {i}: {answer}\n")
        f.write("-" * 50 + "\n\n")

Question 1: 1. How does the SDG plan’s emphasis on equitable resource distribution address the issue of food availability mentioned in the passage?  
2. In what ways does the passage link the progress in reducing extreme poverty to the broader goal of ending hunger and malnutrition?  
3. Explain how the SDG targets for clean water and sanitation interact with the projected increase in water scarcity by 2050.  
4. Discuss the role of technological innovation, as described in the passage, in achieving both economic growth and environmental sustainability.  
5. Analyze how the SDG framework seeks to balance the need for industrialization with the protection of terrestrial ecosystems.  

6. Imagine you are given a dataset of countries with their poverty rates and populations. Write a Python function that returns the top five countries that would benefit most from the SDG “End extreme poverty” goal, defined as the highest product of poverty_rate × population.  
7. Suppose you need to extrac

In [ ]:
# # Answer each question and save to a file
# for question in all_questions:
#     print("Question: ", question)
#     answer = answer_generation_chain.invoke(question)
#     print("Answer: ", answer)
#     print("--------------------------------------------------\\n\\n")
#     # Save answer to file
#     with open("answers.txt", "a") as f:
#         f.write("Question: " + question + "\\n")
#         f.write("Answer: " + answer + "\\n")
#         f.write("--------------------------------------------------\\n\\n")



Question:  1. How does the SDG plan’s emphasis on equitable resource distribution address the issue of food availability mentioned in the passage?  
2. In what ways does the passage link the progress in reducing extreme poverty to the broader goal of ending hunger and malnutrition?  
3. Explain how the SDG targets for clean water and sanitation interact with the projected increase in water scarcity by 2050.  
4. Discuss the role of technological innovation, as described in the passage, in achieving both economic growth and environmental sustainability.  
5. Analyze how the SDG framework seeks to balance the need for industrialization with the protection of terrestrial ecosystems.  

6. Imagine you are given a dataset of countries with their poverty rates and populations. Write a Python function that returns the top five countries that would benefit most from the SDG “End extreme poverty” goal, defined as the highest product of poverty_rate × population.  
7. Suppose you need to extract